# 3. Descriptive Statistics
Updated analysis of `BA_overseas_data.xlsx`. One valid BA job ID is one posting. Non-posting footer rows are excluded. Countries, percentages, dates and interpretations are calculated from the current workbook.

**Run all cells from the top.** Dependencies: pandas, numpy, openpyxl and a Jupyter environment. Place the workbook beside this notebook or in `../data/cleaned/`. Industry case variants and equivalent bachelor labels are standardized in memory. The source workbook is not modified.


## 3.1 Load and prepare the updated dataset
AI is recorded in `ai_skill_explicit` (0/1) and `ai_skill_level` (0/2/3). These variables have different meanings.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

import pandas as pd

raw_df = pd.read_excel("../data/cleaned/BA_overseas_data.xlsx")
raw_df.columns = raw_df.columns.astype(str).str.strip()
raw_columns = raw_df.columns.tolist()
required_columns = [
    'Job_id', 'Country', 'Job_Title', 'Company_Name', 'Industry', 'Source', 'Job_Url',
    'Date_Posted', 'Date_Collected', 'Salary_Min', 'Salary_Max',
    'Salary_Currency', 'Experience_Min', 'Experience_Max', 'Education',
    'Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI',
    'ai_skill_explicit', 'ai_skill_level', 'JD_text',
]
missing_columns = sorted(set(required_columns) - set(raw_df.columns))
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

# The workbook ends with a total, an empty row, and a coding note.
# These are not vacancies and must never enter a posting denominator.
valid_job = raw_df['Job_id'].astype('string').str.strip().str.fullmatch(r'BA\d+', na=False)
excluded_rows = raw_df.loc[~valid_job].copy()
df = raw_df.loc[valid_job].copy().reset_index(drop=True)
if df.empty:
    raise ValueError('No valid BA job IDs were found.')
df['Job_id'] = df['Job_id'].astype('string').str.strip()
if df['Job_id'].duplicated().any():
    raise ValueError('Duplicate Job_id values found. Review them before analysis.')

for col in ['Country', 'Industry', 'Education', 'Salary_Currency', 'Source']:
    df[col] = df[col].astype('string').str.strip().replace('', pd.NA)
country_map = {
    'united states': 'United States', 'united kingdom': 'United Kingdom',
    'singapore': 'Singapore', 'hong kong': 'Hong Kong', 'canada': 'Canada',
    'australia': 'Australia', 'ireland': 'Ireland',
}
df['Country'] = df['Country'].str.casefold().map(country_map).fillna(df['Country'])
industry_map = {
    'financial services': 'Financial Services', 'technology': 'Technology',
    'industiral': 'Industrial', 'industrial': 'Industrial',
    'education': 'Education', 'other': 'Other', 'unknown': 'Unknown',
}
df['Industry_Clean'] = df['Industry'].str.casefold().map(industry_map).fillna(df['Industry']).fillna('Unknown')
education_map = {
    "bachelor's": "Bachelor's", 'bachelor': "Bachelor's",
    'bachelor’s': "Bachelor's", 'not specified': 'Not Specified',
    'third level degree': 'Third level degree',
}
df['Education_Clean'] = df['Education'].str.casefold().map(education_map).fillna(df['Education']).fillna('Not Specified')

for col in ['Date_Posted', 'Date_Collected']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
for col in ['Experience_Min', 'Experience_Max', 'Salary_Min', 'Salary_Max']:
    df[col + '_num'] = pd.to_numeric(df[col], errors='coerce')
    if df[col + '_num'].lt(0).any():
        raise ValueError(f'Negative values in {col}; review the source.')

# Explicit AI mentions replace the old AI_tools indicator.
# The separate level variable is ordinal and is NOT a binary skill.
skill_columns = ['Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI', 'ai_skill_explicit']
skill_labels = {
    'Excel': 'Excel', 'SQL': 'SQL', 'Python': 'Python', 'R': 'R',
    'Tableau': 'Tableau', 'Power_BI': 'Power BI',
    'ai_skill_explicit': 'AI skills explicitly mentioned',
}
for col in skill_columns:
    values = pd.to_numeric(df[col], errors='coerce')
    bad = df[col].notna() & ~values.isin([0, 1])
    if bad.any():
        raise ValueError(f'{col} must contain 0, 1 or missing values: {df.loc[bad, "Job_id"].tolist()}')
    df[col] = values.astype('Int64')
levels = pd.to_numeric(df['ai_skill_level'], errors='coerce')
if (df['ai_skill_level'].notna() & ~levels.isin([0, 2, 3])).any():
    raise ValueError('AI levels must be 0 = no mention, 2 = preferred, or 3 = required.')
df['ai_skill_level'] = levels.astype('Int64')
ai_level_labels = {0: 'No AI mention', 2: 'AI preferred', 3: 'AI required'}
df['AI_Level_Label'] = df['ai_skill_level'].map(ai_level_labels).astype('string').fillna('Not specified')

# Preserve suspect values for audit. Exclude inverted ranges only from the
# additional salary summary with valid bounds; never guess a corrected salary.
inverted_salary = (df['Salary_Min_num'].notna() & df['Salary_Max_num'].notna()
                   & df['Salary_Min_num'].gt(df['Salary_Max_num']))
df['Salary_Range_Invalid'] = inverted_salary
valid_salary_pair = (df[['Salary_Min_num', 'Salary_Max_num']].notna().all(axis=1)
                     & ~inverted_salary)
df['Salary_Midpoint'] = ((df['Salary_Min_num'] + df['Salary_Max_num']) / 2).where(valid_salary_pair)
df['Number_of_Skills'] = df[skill_columns].sum(axis=1, min_count=len(skill_columns))
ai_comparable = df[['ai_skill_explicit', 'ai_skill_level']].notna().all(axis=1)
ai_inconsistent = ai_comparable & df['ai_skill_explicit'].ne(df['ai_skill_level'].isin([2, 3]).astype(int))

# Flags are for review, not proof that a source value is wrong.
date_after_collection = df['Date_Posted'].gt(df['Date_Collected'])
quality_summary = pd.DataFrame({
    'Check': ['Non-posting rows excluded', 'Duplicate job URLs',
              'Posting date after collection date', 'Minimum salary exceeds maximum',
              'AI explicit and level disagree'],
    'Count': [len(excluded_rows), df['Job_Url'].dropna().duplicated().sum(),
              date_after_collection.sum(), inverted_salary.sum(), ai_inconsistent.sum()],
})

N = len(df)
country_sizes = df['Country'].fillna('Not Specified').value_counts()
country_order = country_sizes.index.tolist()
industry_sizes = df['Industry_Clean'].value_counts()
industry_order = industry_sizes.index.tolist()

def frequency_table(series, label):
    result = series.fillna('Not Specified').value_counts(dropna=False).rename_axis(label).reset_index(name='Job_Postings')
    result['Percentage'] = (result['Job_Postings'] / N * 100).round(1)
    return result

def skill_rates_by(group):
    # Missing indicators are excluded per skill, rather than counted as zero.
    grouping = df[group].fillna('Not Specified')
    return df.groupby(grouping)[skill_columns].mean().mul(100)

print(f'Dataset: {data_path.name}')
print(f'Excel rows: {len(raw_df)}; valid job postings: {N}; excluded non-posting rows: {len(excluded_rows)}')
print(f'Countries: {df["Country"].nunique()}; standardized industries: {df["Industry_Clean"].nunique()}')
print('Collection period:', df['Date_Collected'].min().date(), 'to', df['Date_Collected'].max().date())


Dataset: BA_overseas_data.xlsx
Excel rows: 199; valid job postings: 196; excluded non-posting rows: 3
Countries: 7; standardized industries: 5
Collection period: 2026-09-15 to 2026-10-27


## 3.2 Basic dataset information and quality checks
Inspect excluded rows and quality flags before interpreting the summaries. A posting date after its collection date needs verification; the code does not guess a replacement date.


In [3]:
basic_summary = pd.DataFrame({
    'Metric': ['Valid job postings', 'Original variables', 'Countries', 'Standardized industries', 'Unique job titles', 'Unique companies'],
    'Value': [N, len(raw_columns), df['Country'].nunique(), df['Industry_Clean'].nunique(), df['Job_Title'].nunique(), df['Company_Name'].nunique()],
})
display(basic_summary)
display(quality_summary)
print('Excluded Excel rows (header is row 1):', (excluded_rows.index + 2).tolist())
display(excluded_rows.dropna(axis=1, how='all'))
display(df.loc[inverted_salary, ['Job_id', 'Country', 'Salary_Min', 'Salary_Max', 'Salary_Currency']])
display(df.loc[date_after_collection, ['Job_id', 'Date_Posted', 'Date_Collected']])
print('Source columns:', raw_columns)


,Metric,Value
0,Valid job postings,196
1,Original variables,26
2,Countries,7
3,Standardized industries,5
4,Unique job titles,117
5,Unique companies,166


,Check,Count
0,Non-posting rows excluded,3
1,Duplicate job URLs,0
2,Posting date after collection date,51
3,Minimum salary exceeds maximum,1
4,AI explicit and level disagree,0


Excluded Excel rows (header is row 1): [198, 199, 200]


,Job_id,ai_skill_explicit
196,NaN,23.0
197,NaN,NaN
198,Note:ai_skill_explicit is a binary indicator (...,NaN


,Job_id,Country,Salary_Min,Salary_Max,Salary_Currency
171,BA172,Australia,85000.0,10000.0,AUD


,Job_id,Date_Posted,Date_Collected
3,BA004,2026-09-17,2026-09-15
4,BA005,2026-09-16,2026-09-15
8,BA009,2026-09-18,2026-09-15
10,BA011,2026-09-21,2026-09-15
11,BA012,2026-09-19,2026-09-15
14,BA015,2026-09-17,2026-09-15
15,BA016,2026-09-17,2026-09-15
20,BA021,2026-09-19,2026-09-15
22,BA023,2026-09-18,2026-09-15
23,BA024,2026-09-17,2026-09-15


Source columns: ['Job_id', 'Country', 'City', 'Job_Title', 'Company_Name', 'Industry', 'Source', 'Job_Url', 'Date_Posted', 'Date_Collected', 'Salary_Min', 'Salary_Max', 'Salary_Currency', 'Experience_Min', 'Experience_Max', 'Education', 'Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI', 'ai_skill_explicit', 'ai_skill_level', 'JD_text', 'Collector']


## 3.3 Job postings by country
Counts describe the collected sample. They do not estimate the total number of vacancies in each labor market.


In [4]:
country_summary = frequency_table(df['Country'], 'Country')
display(country_summary)


,Country,Job_Postings,Percentage
0,United Kingdom,44,22.4
1,United States,40,20.4
2,Hong Kong,31,15.8
3,Canada,27,13.8
4,Australia,24,12.2
5,Singapore,15,7.7
6,Ireland,15,7.7


## 3.4 Job postings by industry
Financial Services capitalization variants are combined and Industiral is displayed as Industrial. Raw labels are retained in `Industry`.


In [5]:
industry_summary = frequency_table(df['Industry_Clean'], 'Industry')
display(industry_summary)
print('Original labels:')
display(df['Industry'].value_counts(dropna=False).to_frame('Job_Postings'))


,Industry,Job_Postings,Percentage
0,Technology,93,47.4
1,Financial Services,82,41.8
2,Other,19,9.7
3,Industrial,1,0.5
4,Education,1,0.5


Original labels:


,Job_Postings
Industry,
Technology,93
Financial Services,81
Other,19
Industiral,1
Education,1
Financial services,1


## 3.5 Education requirements
Bachelor and Bachelor's are combined. Third level degree is retained separately. Not Specified does not mean that no qualification is required.


In [6]:
education_summary = frequency_table(df['Education_Clean'], 'Education')
display(education_summary)
display(pd.crosstab(df['Country'], df['Education_Clean'], normalize='index').mul(100).round(1))


,Education,Job_Postings,Percentage
0,Not Specified,118,60.2
1,Bachelor's,77,39.3
2,Third level degree,1,0.5


Education_Clean,Bachelor's,Not Specified,Third level degree
Country,,,
Australia,16.7,83.3,0.0
Canada,37.0,63.0,0.0
Hong Kong,51.6,48.4,0.0
Ireland,26.7,73.3,0.0
Singapore,40.0,60.0,0.0
United Kingdom,18.2,79.5,2.3
United States,72.5,27.5,0.0


## 3.6 Experience requirements
Not Specified is treated as missing, not zero. Means and medians use numeric observations only. Country comparisons show their available sample sizes.


In [7]:
experience_summary = df[['Experience_Min_num', 'Experience_Max_num']].describe().T
experience_summary['Missing_Count'] = N - experience_summary['count']
experience_summary['Missing_Rate'] = (experience_summary['Missing_Count'] / N * 100).round(1)
display(experience_summary)
experience_counts = df['Experience_Min_num'].value_counts().sort_index().rename_axis('Minimum_Years').to_frame('Job_Postings')
display(experience_counts)
experience_by_country = df.groupby('Country')['Experience_Min_num'].agg(['count', 'mean', 'median', 'min', 'max']).round(2)
display(experience_by_country)


,count,mean,std,min,25%,50%,75%,max,Missing_Count,Missing_Rate
Experience_Min_num,133.0,4.142857,2.326211,0.0,3.0,4.0,5.0,21.0,63.0,32.1
Experience_Max_num,32.0,5.906250,2.387594,3.0,5.0,5.0,7.0,15.0,164.0,83.7


,Job_Postings
Minimum_Years,
0.0,1
1.0,6
2.0,13
3.0,45
4.0,12
5.0,40
6.0,2
7.0,5
8.0,6


,count,mean,median,min,max
Country,,,,,
Australia,16,3.31,3.0,1.0,7.0
Canada,23,4.00,3.0,1.0,8.0
Hong Kong,25,3.24,3.0,1.0,5.0
Ireland,9,4.67,5.0,2.0,8.0
Singapore,15,4.87,5.0,3.0,10.0
United Kingdom,14,3.86,4.5,1.0,7.0
United States,31,5.03,5.0,0.0,21.0


## 3.7 Salary information
Currencies are summarized separately. The first table preserves all recorded endpoints, including flagged ranges. The second excludes rows whose minimum exceeds their maximum. Other extremes are shown for review and retained; no salary is silently corrected. Currency grouping alone does not verify consistent pay periods, so these summaries are not cross-country pay rankings.


In [8]:
salary_fields = ['Salary_Min_num', 'Salary_Max_num']
salary_summary = df.groupby('Salary_Currency')[salary_fields].agg(['count', 'mean', 'median', 'min', 'max']).round(2)
print('All recorded salary endpoints')
display(salary_summary)
salary_summary_valid_bounds = df.loc[~inverted_salary].groupby('Salary_Currency')[salary_fields].agg(['count', 'mean', 'median', 'min', 'max']).round(2)
print('Salary endpoints excluding inverted ranges')
display(salary_summary_valid_bounds)
salary_availability = pd.DataFrame({
    'Variable': ['Salary_Min', 'Salary_Max'],
    'Available': [df[col].notna().sum() for col in salary_fields],
})
salary_availability['Missing'] = N - salary_availability['Available']
salary_availability['Missing_Rate'] = (salary_availability['Missing'] / N * 100).round(1)
display(salary_availability)
print('Largest recorded salary maxima by currency (review values and pay periods)')
display(df.loc[df['Salary_Max_num'].notna()].sort_values('Salary_Max_num', ascending=False).groupby('Salary_Currency', sort=False).head(1)[['Job_id', 'Salary_Currency', 'Salary_Min', 'Salary_Max']])


All recorded salary endpoints


Salary_Min_num                                          \
                         count       mean    median      min       max   
Salary_Currency                                                          
AUD                          3  111749.67  108571.0  85000.0  141678.0   
CAD                         16   74629.75   73500.0  44951.0  120000.0   
EUR                          1   70000.00   70000.0  70000.0   70000.0   
GBP                          5   46240.00   47000.0  42000.0   50000.0   
HKD                          0        NaN       NaN      NaN       NaN   
SGD                          0        NaN       NaN      NaN       NaN   
USD                         31   86473.06   80000.0  57000.0  146670.0   

                Salary_Max_num                                           
                         count       mean    median      min        max  
Salary_Currency                                                          
AUD                          3   96190.00  119893.0  10000.0   158677.0  
CAD                         16  193414.48  105456.0  85000.0  1550000.0  
EUR                          1   75000.00   75000.0  75000.0    75000.0  
GBP                          5   60400.00   56000.0  50000.0    85000.0  
HKD                          0        NaN       NaN      NaN        NaN  
SGD                          0        NaN       NaN      NaN        NaN  
USD                         31  133141.65  125000.0  85000.0   249330.0

Salary endpoints excluding inverted ranges


Salary_Min_num                                           \
                         count       mean    median       min       max   
Salary_Currency                                                           
AUD                          2  125124.50  125124.5  108571.0  141678.0   
CAD                         16   74629.75   73500.0   44951.0  120000.0   
EUR                          1   70000.00   70000.0   70000.0   70000.0   
GBP                          5   46240.00   47000.0   42000.0   50000.0   
HKD                          0        NaN       NaN       NaN       NaN   
SGD                          0        NaN       NaN       NaN       NaN   
USD                         31   86473.06   80000.0   57000.0  146670.0   

                Salary_Max_num                                            
                         count       mean    median       min        max  
Salary_Currency                                                           
AUD                          2  139285.00  139285.0  119893.0   158677.0  
CAD                         16  193414.48  105456.0   85000.0  1550000.0  
EUR                          1   75000.00   75000.0   75000.0    75000.0  
GBP                          5   60400.00   56000.0   50000.0    85000.0  
HKD                          0        NaN       NaN       NaN        NaN  
SGD                          0        NaN       NaN       NaN        NaN  
USD                         31  133141.65  125000.0   85000.0   249330.0

,Variable,Available,Missing,Missing_Rate
0,Salary_Min,56,140,71.4
1,Salary_Max,56,140,71.4


Largest recorded salary maxima by currency (review values and pay periods)


,Job_id,Salary_Currency,Salary_Min,Salary_Max
154,BA155,CAD,80000.0,1550000.0
30,BA031,USD,146670.0,249330.0
178,BA179,AUD,141678.0,158677.0
62,BA063,GBP,NaN,85000.0
195,BA196,EUR,70000.0,75000.0


## 3.8 Technical skill requirements
The seven indicators include the new explicit AI requirement variable. Mention rates use nonmissing indicators per skill; `Share_All_Postings` is also reported. A missing indicator is never automatically changed to zero.


In [9]:
skill_summary = pd.DataFrame({
    'Skill_Column': skill_columns,
    'Skill': [skill_labels[col] for col in skill_columns],
    'Number_of_Jobs': [int(df[col].sum()) for col in skill_columns],
    'Available': [df[col].notna().sum() for col in skill_columns],
})
skill_summary['Missing'] = N - skill_summary['Available']
skill_summary['Mention_Rate'] = (skill_summary['Number_of_Jobs'] / skill_summary['Available'].replace(0, np.nan) * 100).round(1)
skill_summary['Share_All_Postings'] = (skill_summary['Number_of_Jobs'] / N * 100).round(1)
skill_summary = skill_summary.sort_values('Number_of_Jobs', ascending=False, kind='stable').reset_index(drop=True)
display(skill_summary)
display(skill_rates_by('Country').round(1))
print('Industry sample sizes')
display(industry_sizes.to_frame('Job_Postings'))
display(skill_rates_by('Industry_Clean').round(1))


,Skill_Column,Skill,Number_of_Jobs,Available,Missing,Mention_Rate,Share_All_Postings
0,Excel,Excel,44,196,0,22.4,22.4
1,SQL,SQL,42,196,0,21.4,21.4
2,ai_skill_explicit,AI skills explicitly mentioned,23,196,0,11.7,11.7
3,Power_BI,Power BI,19,196,0,9.7,9.7
4,Tableau,Tableau,6,196,0,3.1,3.1
5,Python,Python,5,196,0,2.6,2.6
6,R,R,0,196,0,0.0,0.0


,Excel,SQL,Python,R,Tableau,Power_BI,ai_skill_explicit
Country,,,,,,,
Australia,12.5,16.7,4.2,0.0,0.0,8.3,12.5
Canada,25.9,14.8,0.0,0.0,0.0,11.1,18.5
Hong Kong,19.4,29.0,6.5,0.0,3.2,16.1,22.6
Ireland,0.0,13.3,0.0,0.0,0.0,0.0,6.7
Singapore,33.3,26.7,6.7,0.0,0.0,0.0,6.7
United Kingdom,18.2,22.7,2.3,0.0,4.5,13.6,6.8
United States,37.5,22.5,0.0,0.0,7.5,7.5,7.5


Industry sample sizes


,Job_Postings
Industry_Clean,
Technology,93
Financial Services,82
Other,19
Industrial,1
Education,1


,Excel,SQL,Python,R,Tableau,Power_BI,ai_skill_explicit
Industry_Clean,,,,,,,
Education,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Financial Services,19.5,22.0,3.7,0.0,4.9,9.8,18.3
Industrial,0.0,0.0,0.0,0.0,0.0,100.0,100.0
Other,10.5,21.1,0.0,0.0,0.0,21.1,0.0
Technology,28.0,21.5,2.2,0.0,2.2,6.5,7.5


## 3.9 Explicit AI skills and requirement intensity
`ai_skill_level`: 0 = no AI mention, 2 = preferred, 3 = required. These are category codes, not a continuous score; their mean is not reported. Conditional percentages identify the split among postings with explicit AI mentions.


In [10]:
ai_summary = frequency_table(df['AI_Level_Label'], 'AI_Level')
ai_mentions = int(df['ai_skill_explicit'].eq(1).sum())
ai_summary['Share_of_AI_Mentions'] = np.where(
    ai_summary['AI_Level'].isin(['AI preferred', 'AI required']),
    (ai_summary['Job_Postings'] / ai_mentions * 100).round(1) if ai_mentions else np.nan,
    np.nan,
)
display(ai_summary)
ai_by_country = pd.crosstab(df['Country'], df['AI_Level_Label']).reindex(columns=['No AI mention', 'AI preferred', 'AI required', 'Not specified'], fill_value=0)
ai_by_country_pct = ai_by_country.div(ai_by_country.sum(axis=1), axis=0).mul(100)
display(ai_by_country)
display(ai_by_country_pct.round(1))


,AI_Level,Job_Postings,Percentage,Share_of_AI_Mentions
0,No AI mention,173,88.3,NaN
1,AI required,13,6.6,56.5
2,AI preferred,10,5.1,43.5


AI_Level_Label,No AI mention,AI preferred,AI required,Not specified
Country,,,,
Australia,21,1,2,0
Canada,22,3,2,0
Hong Kong,24,2,5,0
Ireland,14,1,0,0
Singapore,14,0,1,0
United Kingdom,41,0,3,0
United States,37,3,0,0


AI_Level_Label,No AI mention,AI preferred,AI required,Not specified
Country,,,,
Australia,87.5,4.2,8.3,0.0
Canada,81.5,11.1,7.4,0.0
Hong Kong,77.4,6.5,16.1,0.0
Ireland,93.3,6.7,0.0,0.0
Singapore,93.3,0.0,6.7,0.0
United Kingdom,93.2,0.0,6.8,0.0
United States,92.5,7.5,0.0,0.0


## 3.10 Missing value summary
This table measures analytic missingness after nonnumeric experience and salary values are coerced. Education coded Not Specified is counted separately. Raw and derived copies are not counted twice.


In [11]:
analytic_columns = ['Country', 'Industry_Clean', 'Education_Clean', 'Date_Posted', 'Date_Collected', 'Salary_Currency', 'Experience_Min_num', 'Experience_Max_num', 'Salary_Min_num', 'Salary_Max_num'] + skill_columns + ['ai_skill_level']
missing_summary = pd.DataFrame({
    'Missing_Count': df[analytic_columns].isna().sum(),
    'Missing_Percentage': (df[analytic_columns].isna().mean() * 100).round(1),
}).sort_values('Missing_Count', ascending=False)
display(missing_summary.loc[missing_summary['Missing_Count'].gt(0)])
print('Education Not Specified:', df['Education_Clean'].eq('Not Specified').sum())


,Missing_Count,Missing_Percentage
Experience_Max_num,164,83.7
Salary_Min_num,140,71.4
Salary_Max_num,140,71.4
Experience_Min_num,63,32.1


Education Not Specified: 118


## 3.11 Key statistics summary


In [11]:
most_requested_skill = skill_summary.loc[0, 'Skill']
key_statistics = pd.DataFrame({
    'Metric': ['Total valid job postings', 'Number of countries', 'Number of standardized industries',
               'Median stated minimum experience in years', 'Salary minimum available', 'Salary maximum available',
               'Most frequently identified skill', 'Explicit AI mentions', 'AI preferred', 'AI required'],
    'Value': [N, df['Country'].nunique(), df['Industry_Clean'].nunique(), df['Experience_Min_num'].median(),
              df['Salary_Min_num'].notna().sum(), df['Salary_Max_num'].notna().sum(), most_requested_skill,
              ai_mentions, df['ai_skill_level'].eq(2).sum(), df['ai_skill_level'].eq(3).sum()],
})
display(key_statistics)


,Metric,Value
0,Total valid job postings,196
1,Number of countries,7
2,Number of standardized industries,5
3,Median stated minimum experience in years,4.0
4,Salary minimum available,56
5,Salary maximum available,56
6,Most frequently identified skill,Excel
7,Explicit AI mentions,23
8,AI preferred,10
9,AI required,13


## 3.12 Interpretation of the current results
The narrative below is calculated each time the notebook runs.


In [12]:
print(f'The analysis contains {N} valid postings across {df["Country"].nunique()} markets. '
      f'{len(excluded_rows)} non-posting Excel rows are excluded from every denominator.')
print('Country composition: ' + '; '.join(f'{country}: {count} ({count/N:.1%})' for country, count in country_sizes.items()) + '.')
print('Skill mentions: ' + '; '.join(f'{row.Skill}: {row.Number_of_Jobs} ({row.Mention_Rate:.1f}% of available indicators)' for row in skill_summary.itertuples()) + '.')
print(f'Numeric minimum experience is available for {df["Experience_Min_num"].notna().sum()} postings; '
      f'the median is {df["Experience_Min_num"].median():g} years. '
      f'Education is unspecified in {df["Education_Clean"].eq("Not Specified").sum()} postings.')
print(f'AI skills are explicitly mentioned in {ai_mentions} postings ({ai_mentions/N:.1%}); '
      f'{df["ai_skill_level"].eq(2).sum()} classify AI as preferred and '
      f'{df["ai_skill_level"].eq(3).sum()} classify it as required.')
print('These are descriptive patterns in collected advertisements, not population estimates, causal effects or changes over time. '
      'A zero skill indicator means no coded mention. Salary comparisons require checks of disclosure, ranges and pay periods.')


The analysis contains 196 valid postings across 7 markets. 3 non-posting Excel rows are excluded from every denominator.
Country composition: United Kingdom: 44 (22.4%); United States: 40 (20.4%); Hong Kong: 31 (15.8%); Canada: 27 (13.8%); Australia: 24 (12.2%); Singapore: 15 (7.7%); Ireland: 15 (7.7%).
Skill mentions: Excel: 44 (22.4% of available indicators); SQL: 42 (21.4% of available indicators); AI skills explicitly mentioned: 23 (11.7% of available indicators); Power BI: 19 (9.7% of available indicators); Tableau: 6 (3.1% of available indicators); Python: 5 (2.6% of available indicators); R: 0 (0.0% of available indicators).
Numeric minimum experience is available for 133 postings; the median is 4 years. Education is unspecified in 118 postings.
AI skills are explicitly mentioned in 23 postings (11.7%); 10 classify AI as preferred and 13 classify it as required.
These are descriptive patterns in collected advertisements, not population estimates, causal effects or changes over t